# 🚀 Notebook 3 — RustCPG-Detect: Full CPG-Enhanced Pipeline
**The main notebook.** This implements our complete RustCPG-Detect system.

## What this notebook does
1. Loads the CPG dataset from Kaggle
2. Extracts our novel **5,093-dim CPG feature vector** per function
3. Runs **Random Forest feature selection** (top 2,000 features)
4. Trains **Binary CatBoost** with CPG features + threshold tuning
5. Evaluates with **3-Fold Cross-Validation**

## Key Results
| Metric | Value |
|---|---|
| Binary Accuracy | **93.49%** |
| Macro F1 | **0.896** |
| 3-Fold CV | **92.47% ± 0.47%** |
| Optimal Threshold | **0.71** |
| Vuln Recall | **96.8%** |

> **Dataset**: [Kaggle — kaarthikeyaganji/cid-i2](https://www.kaggle.com/datasets/kaarthikeyaganji/cid-i2)


In [ ]:
!pip install catboost torch-geometric torch-scatter -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, os
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier, Pool
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

DRIVE_DIR = "/kaggle/working"
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)

CLASS_NAMES_5 = {0:'Safe', 1:'UAF', 2:'Data Race',
                  3:'Int Overflow', 4:'Mem Corrupt'}
CLASS_NAMES_BIN = {0:'Safe', 1:'Vulnerable'}
NUM_CLASSES = 5

print("Setup complete ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.1 MB/s eta 0:00:0000:01
Setup complete ✅


## Load Dataset
Load the pre-built CPG dataset. Each sample is a PyTorch Geometric `Data` object with:
- `x`: node feature matrix `[num_blocks, 835]` (768 BERT + 67 structural)
- `edge_index`: CPG edges `[2, num_edges]`  
- `edge_attr`: edge types `[num_edges]` (0=control flow, 1=data flow)
- `y`: vulnerability class label `[1]`


In [ ]:
dataset = torch.load(
    "/kaggle/input/datasets/kaarthikeyaganji/cid-i2/embedded_dataset_balanced.pt",
    weights_only=False
)

counts = Counter([d.y.item() for d in dataset])
print(f"Loaded {len(dataset)} samples")
print("Class distribution:")
for cls, name in CLASS_NAMES_5.items():
    print(f"  {name}: {counts[cls]}")

# Quick structural check
d0 = dataset[0]
print(f"\nNode features: {d0.x.shape}  (BERT:768 + Struct:67)")
print(f"Edge index:    {d0.edge_index.shape}")
print(f"Edge attr:     {d0.edge_attr.shape}  (values: {d0.edge_attr.flatten().unique().tolist()})")

Loaded 32510 samples
Class distribution:
  Safe: 6502
  UAF: 6502
  Data Race: 6502
  Int Overflow: 6502
  Mem Corrupt: 6502

Node features: torch.Size([7, 835])  (BERT:768 + Struct:67)
Edge index:    torch.Size([2, 11])
Edge attr:     torch.Size([11, 1])  (values: [0.0, 1.0])


## CPG Feature Extraction (5,093-dim)
Our novel feature extraction combines 7 pooling strategies across BERT and structural features.

| Component | Dims | Description |
|---|---|---|
| BERT 4-Pool | 3,072 | mean, max, std, min of 768-dim BERT embeddings |
| Structural 4-Pool | 268 | mean, max, std, sum of 67-dim structural features |
| Block Extremes | 201 | hotspot block + entry block + exit block features |
| Sequence Deltas | 1,536 | mean & max of inter-block BERT differences |
| Graph Topology | 8 | nodes, edges, density, CFG/DFG ratios, max degree |
| Cosine Similarity | 4 | mean, std, min, max of inter-block semantic similarity |
| Diversity Scores | 4 | structural diversity statistics |
| **Total** | **5,093** | |


In [ ]:
def extract_cpg_features(dataset, mode='binary'):
    """
    Novel CPG-aware feature extraction.
    Combines BERT pooling + structural pooling +
    ordered block features + graph topology.

    mode='binary' → also returns binary labels
    mode='multi'  → returns 5-class labels
    """
    X_list, y_bin, y_multi = [], [], []

    for d in dataset:
        x      = d.x.numpy()
        bert   = x[:, :768]
        struct = x[:, 768:]
        n      = x.shape[0]

        # ── 1. BERT: 4-pool (3072-dim) ──────────────────────────────
        b_mean = bert.mean(0)
        b_max  = bert.max(0)
        b_std  = bert.std(0)
        b_min  = bert.min(0)

        # ── 2. Structural: 4-pool (268-dim) ─────────────────────────
        s_mean = struct.mean(0)
        s_max  = struct.max(0)
        s_std  = struct.std(0)
        s_sum  = struct.sum(0)

        # ── 3. Block extremes — hotspot + entry + exit (201-dim) ────
        # (Novel: captures vulnerability hotspot block explicitly)
        hotspot   = struct[struct.sum(1).argmax()]
        first_blk = struct[0]
        last_blk  = struct[-1]

        # ── 4. Sequence features — ordered block deltas (1536-dim) ──
        # (Novel: captures IR transformation flow, not just bag-of-blocks)
        if n > 1:
            deltas     = np.diff(bert, axis=0)
            delta_mean = deltas.mean(0)   # 768
            delta_max  = np.abs(deltas).max(0)  # 768
        else:
            delta_mean = np.zeros(768)
            delta_max  = np.zeros(768)

        # ── 5. Graph topology (8-dim) ────────────────────────────────
        if d.edge_index is not None and d.edge_index.shape[1] > 0:
            ei  = d.edge_index.numpy()
            ea  = d.edge_attr.numpy().flatten()
            ne  = ei.shape[1]
            cfg = float((ea == 0).sum())
            dfg = float((ea == 1).sum())
            out_deg = np.bincount(ei[0], minlength=n).astype(float)
            in_deg  = np.bincount(ei[1], minlength=n).astype(float)
            topo = np.array([
                float(n), float(ne), float(ne)/max(n,1),
                cfg/max(ne,1), dfg/max(ne,1),
                out_deg.max(), in_deg.max(),
                float(dfg > cfg)
            ])
        else:
            topo = np.zeros(8)

        # ── 6. Inter-block cosine similarity (4-dim) ─────────────────
        if n > 1:
            norms = np.linalg.norm(bert, axis=1, keepdims=True) + 1e-8
            sims  = ((bert/norms)[:-1] * (bert/norms)[1:]).sum(1)
            sim_f = np.array([sims.mean(), sims.std(), sims.min(), sims.max()])
        else:
            sim_f = np.array([1., 0., 1., 1.])

        # ── 7. Diversity scores (4-dim) ───────────────────────────────
        diversity = np.array([
            float(struct.max()), float(struct.sum()),
            float(struct.std()), float((struct > 0).mean())
        ])

        feat = np.concatenate([
            b_mean, b_max, b_std, b_min,        # 3072
            s_mean, s_max, s_std, s_sum,          # 268
            hotspot, first_blk, last_blk,          # 201
            delta_mean, delta_max,                  # 1536
            topo, sim_f, diversity                  # 16
        ])  # Total: 5093-dim

        X_list.append(feat)
        label = d.y.item()
        y_bin.append(0 if label == 0 else 1)
        y_multi.append(label)

    X = np.stack(X_list)
    return X, np.array(y_bin), np.array(y_multi)

print("Extracting CPG features (this takes ~3 min)...")
X_full, y_bin_full, y_multi_full = extract_cpg_features(dataset)
print(f"Feature shape: {X_full.shape}")
print(f"Binary  — Safe: {(y_bin_full==0).sum()}  Vuln: {(y_bin_full==1).sum()}")
print(f"5-class — {Counter(y_multi_full.tolist())}")


Extracting CPG features (this takes ~3 min)...
Feature shape: (32510, 5093)
Binary  — Safe: 6502  Vuln: 26008
5-class — Counter({1: 6502, 2: 6502, 3: 6502, 4: 6502, 0: 6502})


## Data Splits + Scaling
70% train / 15% val / 15% test — stratified to preserve class balance.
StandardScaler fitted only on training data, then applied to val and test.


In [ ]:
# ── Binary split ──────────────────────────────────────────────────────
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_full, y_bin_full, test_size=0.20, stratify=y_bin_full, random_state=42
)
X_tr_b, X_val_b, y_tr_b, y_val_b = train_test_split(
    X_tr_b, y_tr_b, test_size=0.125, stratify=y_tr_b, random_state=42
)

# ── 5-class split ────────────────────────────────────────────────────
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_full, y_multi_full, test_size=0.20, stratify=y_multi_full, random_state=42
)
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_tr_m, y_tr_m, test_size=0.125, stratify=y_tr_m, random_state=42
)

# ── Scale ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_b_s  = scaler.fit_transform(X_tr_b)
X_val_b_s = scaler.transform(X_val_b)
X_te_b_s  = scaler.transform(X_te_b)
# Use same scaler for 5-class (same feature space)
X_tr_m_s  = scaler.transform(X_tr_m)
X_val_m_s = scaler.transform(X_val_m)
X_te_m_s  = scaler.transform(X_te_m)

print(f"Binary  — Train:{X_tr_b_s.shape} Val:{X_val_b_s.shape} Test:{X_te_b_s.shape}")
print(f"5-class — Train:{X_tr_m_s.shape} Val:{X_val_m_s.shape} Test:{X_te_m_s.shape}")

Binary  — Train:(22757, 5093) Val:(3251, 5093) Test:(6502, 5093)
5-class — Train:(22757, 5093) Val:(3251, 5093) Test:(6502, 5093)


## Random Forest Feature Selection
300-tree RF on the 5,093-dim features to identify the most predictive 2,000.
This reduces noise and improves CatBoost generalisation.


In [ ]:
print("Running RF ceiling check + feature selection...")

rf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_tr_b_s, y_tr_b)
rf_pred = rf.predict(X_te_b_s)
rf_acc  = accuracy_score(y_te_b, rf_pred)
rf_f1   = f1_score(y_te_b, rf_pred, average='macro', zero_division=0)

print(f"\nBINARY RF Ceiling: Acc={rf_acc:.4f} | F1={rf_f1:.4f}")
print(classification_report(y_te_b, rf_pred,
      target_names=['Safe','Vulnerable'], zero_division=0))

# Feature selection — top 2000 by importance
top_k      = np.argsort(rf.feature_importances_)[::-1][:2000]
X_tr_sel   = X_tr_b_s[:, top_k]
X_val_sel  = X_val_b_s[:, top_k]
X_te_sel   = X_te_b_s[:, top_k]

# Also for 5-class
X_tr_sel_m  = X_tr_m_s[:, top_k]
X_val_sel_m = X_val_m_s[:, top_k]
X_te_sel_m  = X_te_m_s[:, top_k]

print(f"\nSelected top 2000 features ✅")
print("PROCEED ✅" if rf_acc >= 0.90 else f"RF at {rf_acc:.2%} — CatBoost will push higher")

Running RF ceiling check + feature selection...

BINARY RF Ceiling: Acc=0.9163 | F1=0.8459
              precision    recall  f1-score   support

        Safe       0.97      0.60      0.74      1300
  Vulnerable       0.91      1.00      0.95      5202

    accuracy                           0.92      6502
   macro avg       0.94      0.80      0.85      6502
weighted avg       0.92      0.92      0.91      6502


Selected top 2000 features ✅
PROCEED ✅


## Core Result — Enhanced Binary CatBoost
Our main result: binary Safe/Vulnerable classification using CPG-enhanced features.
Threshold tuned on validation set via F1-sweep from 0.1 to 0.9.


In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  CORE RESULT — Binary Safe/Vulnerable Classification
#  Target: ≥ 90% Accuracy, ≥ 0.85 Macro F1
# ─────────────────────────────────────────────────────────────────────

binary_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=150,
    verbose=200,
    task_type='GPU'
)

print("Training Enhanced Binary CatBoost (CPG features)...")
binary_model.fit(
    Pool(X_tr_sel, y_tr_b),
    eval_set=Pool(X_val_sel, y_val_b),
    use_best_model=True
)

# Threshold tuning
val_probs = binary_model.predict_proba(X_val_sel)[:, 1]
best_t, best_f = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    p = (val_probs >= t).astype(int)
    f = f1_score(y_val_b, p, average='macro', zero_division=0)
    if f > best_f:
        best_f, best_t = f, t
print(f"\nOptimal threshold: {best_t:.2f} | Val F1: {best_f:.4f}")

# Test evaluation
te_probs      = binary_model.predict_proba(X_te_sel)[:, 1]
y_pred_std    = (te_probs >= 0.50).astype(int)
y_pred_tuned  = (te_probs >= best_t).astype(int)

print("\n" + "="*55)
print("CORE RESULT — Enhanced Binary (CPG Features, Tuned)")
print("="*55)
acc_b = accuracy_score(y_te_b, y_pred_tuned)
f1_b  = f1_score(y_te_b, y_pred_tuned, average='macro', zero_division=0)
print(f"Accuracy: {acc_b:.4f} ({acc_b*100:.2f}%) | Macro F1: {f1_b:.4f}")
print(classification_report(y_te_b, y_pred_tuned,
      target_names=['Safe','Vulnerable'], zero_division=0))

cm_b = confusion_matrix(y_te_b, y_pred_tuned)
tn, fp, fn, tp = cm_b.ravel()
print(f"Confusion Matrix:")
print(f"  TN (Safe→Safe):   {tn}")
print(f"  FP (Safe→Vuln):   {fp}")
print(f"  FN (Vuln→Safe):   {fn}")
print(f"  TP (Vuln→Vuln):   {tp}")
print(f"  Safe Precision:   {tn/(tn+fn):.4f}")
print(f"  Vuln Recall:      {tp/(tp+fn):.4f}")
print(f"  Vuln Precision:   {tp/(tp+fp):.4f}")
print(f"  False Alarm Rate: {fp/(fp+tn):.4f}")

Training Enhanced Binary CatBoost (CPG features)...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8100819	best: 0.8100819 (0)	total: 845ms	remaining: 28m 8s
200:	test: 0.9064866	best: 0.9064866 (200)	total: 55.6s	remaining: 8m 18s
400:	test: 0.9225925	best: 0.9225925 (400)	total: 1m 49s	remaining: 7m 16s
600:	test: 0.9316665	best: 0.9316665 (600)	total: 2m 44s	remaining: 6m 21s
800:	test: 0.9374945	best: 0.9374945 (800)	total: 3m 38s	remaining: 5m 26s
1000:	test: 0.9414988	best: 0.9414988 (1000)	total: 4m 31s	remaining: 4m 31s
1200:	test: 0.9448254	best: 0.9448324 (1196)	total: 5m 26s	remaining: 3m 36s
1400:	test: 0.9473167	best: 0.9473167 (1400)	total: 6m 19s	remaining: 2m 42s
1600:	test: 0.9495218	best: 0.9495218 (1600)	total: 7m 13s	remaining: 1m 48s
1800:	test: 0.9506420	best: 0.9506420 (1800)	total: 8m 7s	remaining: 53.9s
1999:	test: 0.9518889	best: 0.9518889 (1999)	total: 9m	remaining: 0us
bestTest = 0.9518889189
bestIteration = 1999

Optimal threshold: 0.71 | Val F1: 0.8967

CORE RESULT — Enhanced Binary (CPG Features, Tuned)
Accuracy: 0.9349 (93.49%) | Macro F1: 

## 3-Fold Cross-Validation
Robust evaluation across 3 stratified folds to confirm model stability.


In [ ]:
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

skf    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_acc, cv_f1 = [], []
X_cv = X_full[:, top_k]
y_cv = y_bin_full

fold_bar = tqdm(enumerate(skf.split(X_cv, y_cv)), total=3, desc="3-Fold CV", ncols=80)

for fold, (tr_idx, val_idx) in fold_bar:
    sc     = StandardScaler()
    Xf_tr  = sc.fit_transform(X_cv[tr_idx])
    Xf_val = sc.transform(X_cv[val_idx])

    m = CatBoostClassifier(
        iterations=800,           # enough for 92%+
        learning_rate=0.05,       # balanced convergence
        depth=7,                  # keeps model quality
        l2_leaf_reg=3,
        loss_function='Logloss',
        random_seed=fold,
        early_stopping_rounds=50,
        verbose=0,
        task_type='GPU'
    )
    m.fit(Pool(Xf_tr, y_cv[tr_idx]),
          eval_set=Pool(Xf_val, y_cv[val_idx]),
          use_best_model=True)

    preds = (m.predict_proba(Xf_val)[:, 1] >= best_t).astype(int)
    acc   = accuracy_score(y_cv[val_idx], preds)
    f1    = f1_score(y_cv[val_idx], preds, average='macro', zero_division=0)
    cv_acc.append(acc)
    cv_f1.append(f1)
    fold_bar.set_postfix(acc=f"{acc:.4f}", f1=f"{f1:.4f}")
    print(f"  Fold {fold+1}: Acc={acc:.4f}  F1={f1:.4f}")

print(f"\n✅ 3-Fold CV Accuracy : {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"✅ 3-Fold CV Macro F1 : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
print(f"   Base paper reports : 0.982 ± 0.008")

3-Fold CV:   0%|                                          | 0/3 [00:00<?, ?it/s]

  Fold 1: Acc=0.9272  F1=0.8837
  Fold 2: Acc=0.9289  F1=0.8849
  Fold 3: Acc=0.9181  F1=0.8688

✅ 3-Fold CV Accuracy : 0.9247 ± 0.0047
✅ 3-Fold CV Macro F1 : 0.8791 ± 0.0073
   Base paper reports : 0.982 ± 0.008
